# Лабораторная работа 2. Логистическая и линейная регрессии

    ## Данные и протокол

    - **Классификация.** [Optical Recognition of Handwritten Digits](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_digits.html), 1797 изображений размером 8×8, классы от 0 до 9. Практическая задача — распознавание цифр в оцифрованных рукописных формах. Используется поставляемая в sklearn часть UCI Optdigits.
    - **Регрессия.** [Diabetes (Efron и соавт.)](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_diabetes.html), 442 наблюдения и 10 исходных признаков. Цель — численный показатель прогрессирования болезни через год после исходного обследования. Это учебный эксперимент на исторической выборке.

    Для классификации основная метрика **macro F1**, она одинаково взвешивает каждый класс. Accuracy показывает долю верных ответов. Для регрессии основная метрика **MAE** выражает типичную абсолютную ошибку в единицах цели. RMSE сильнее реагирует на крупные промахи. R² показывает качество относительно прогноза средним.

    Разбиение 80/20 с seed 42 одинаково в пяти работах. Для классификации используется стратификация. Гиперпараметры выбираем по среднему результату 5-fold CV на обучающей части. Масштабирование, где оно используется, обучаем заново внутри каждого фолда и на полной обучающей выборке перед оценкой на тесте.

In [1]:
# Импортируем зависимости, задаём разбиение и функции расчёта метрик.
from hashlib import sha256
import json
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.optimize import minimize
from scipy.special import logsumexp, softmax
from sklearn.datasets import load_diabetes, load_digits
from sklearn.ensemble import (GradientBoostingClassifier, GradientBoostingRegressor,
                              RandomForestClassifier, RandomForestRegressor)
from sklearn.linear_model import LinearRegression, LogisticRegression, Ridge
from sklearn.metrics import accuracy_score, f1_score, mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import KFold, StratifiedKFold, train_test_split
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from threadpoolctl import threadpool_limits

SEED = 42

def load_task(classification):
    data = load_digits() if classification else load_diabetes(scaled=False)
    ids = np.arange(len(data.target))
    train, test = train_test_split(ids, test_size=0.2, random_state=SEED,
                                   stratify=data.target if classification else None)
    return data, train, test

def partition(data, train, test, scale):
    X_train, X_test = data.data[train], data.data[test]
    if scale:
        scaler = StandardScaler().fit(X_train)
        X_train, X_test = scaler.transform(X_train), scaler.transform(X_test)
    return X_train, X_test

def metrics(y, prediction, classification):
    if classification:
        return {"macro_f1": f1_score(y, prediction, average="macro", zero_division=0),
                "accuracy": accuracy_score(y, prediction)}
    return {"mae": mean_absolute_error(y, prediction),
            "rmse": mean_squared_error(y, prediction) ** 0.5,
            "r2": r2_score(y, prediction)}

In [2]:
# Проверяем размеры и распределение обучающих данных.
for classification in (True, False):
    data, train, test = load_task(classification)
    print("Классификация" if classification else "Регрессия")
    print(f"Обучение: {len(train)}, тест: {len(test)}, признаков: {data.data.shape[1]}")
    print("Пропуски в обучении:", np.isnan(data.data[train]).sum())
    print(np.bincount(data.target[train]) if classification
          else pd.Series(data.target[train]).describe().round(2).to_string())

Классификация
Обучение: 1437, тест: 360, признаков: 64
Пропуски в обучении: 0
[142 146 142 146 145 145 145 143 139 144]
Регрессия
Обучение: 353, тест: 89, признаков: 10
Пропуски в обучении: 0
count    353.00
mean     153.74
std       78.06
min       25.00
25%       86.00
50%      142.00
75%      214.00
max      346.00


## Бейзлайн sklearn

Фиксируем исходную конфигурацию и измеряем её качество на отложенной части.

In [3]:
# Задаём исходные параметры sklearn и функцию оценки модели.
LAB = 2
def make_model(classification, own, params):
    params = params.copy()
    params.pop("scale")
    if LAB == 1:
        if own:
            return OwnKNN(classification, **params)
        return (KNeighborsClassifier if classification else KNeighborsRegressor)(**params)
    if LAB == 2:
        strength = params["strength"]
        if own:
            return OwnLinear(classification, strength)
        if classification:
            return LogisticRegression(C=strength, max_iter=5000, tol=1e-5)
        return Ridge(alpha=strength) if strength else LinearRegression()
    if LAB == 3:
        if own:
            return OwnTree(classification, **params)
        return (DecisionTreeClassifier if classification else DecisionTreeRegressor)(
            random_state=SEED, **params)
    if LAB == 4:
        if own:
            return OwnForest(classification, **params)
        return (RandomForestClassifier if classification else RandomForestRegressor)(
            random_state=SEED, n_jobs=1, **params)
    if own:
        return OwnBoost(classification, **params)
    return (GradientBoostingClassifier if classification else GradientBoostingRegressor)(
        random_state=SEED, **params)

def evaluate(data, train, test, classification, params, own=False):
    X_train, X_test = partition(data, train, test, params["scale"])
    if LAB == 2 and classification and not params["scale"]:
        # Фиксированная шкала яркости [0, 16] известна из описания данных.
        X_train, X_test = X_train / 16, X_test / 16
    model = make_model(classification, own, params)
    with threadpool_limits(limits=1):
        model.fit(X_train, data.target[train])
        prediction = model.predict(X_test)
    return metrics(data.target[test], prediction, classification)

def validate_candidates(data, train, classification, candidates):
    splitter = (StratifiedKFold if classification else KFold)(
        n_splits=5, shuffle=True, random_state=SEED)
    folds = list(splitter.split(train, data.target[train]))
    metric = "macro_f1" if classification else "mae"
    rows = []
    for hypothesis, params in candidates:
        scores = [evaluate(data, train[fit], train[val], classification, params)[metric]
                  for fit, val in folds]
        rows.append({"гипотеза": hypothesis, "mean": np.mean(scores), "std": np.std(scores)})
    table = pd.DataFrame(rows).set_index("гипотеза")
    winner = table["mean"].idxmax() if classification else table["mean"].idxmin()
    return table, winner, dict(candidates)[winner]

In [4]:
# Обучаем базовые библиотечные модели и считаем метрики.
bases = {}
baseline_scores = {}
for classification in (True, False):
    params = {'scale': False, **{}}
    if LAB == 2:
        params['strength'] = 1.0 if classification else 0.0
    if LAB == 4:
        params['max_features'] = 'sqrt' if classification else None
    bases[classification] = params
    data, train, test = load_task(classification)
    baseline_scores[classification] = evaluate(data, train, test, classification, params)
    print(('Классификация' if classification else 'Регрессия'), baseline_scores[classification])

Классификация {'macro_f1': 0.9577943979141876, 'accuracy': 0.9583333333333334}
Регрессия {'mae': 42.79409467959994, 'rmse': 53.85344583676594, 'r2': 0.45260276297191915}


## Проверка гипотез и улучшенный бейзлайн

Для базовой логистической регрессии делим яркости на 16, чтобы улучшить численную сходимость. Проверяем, помогает ли вместо этого стандартизация по пикселям и другой L2-штраф. Для классификации перебираем C, для регрессии — alpha в ridge-регрессии. Малое C усиливает штраф, большое alpha тоже усиливает штраф.

Исходная конфигурация тоже участвует в CV. Если улучшения не подтверждаются, сохраняем исходные параметры.

In [5]:
# Сравниваем конфигурации на обучающих фолдах, затем оцениваем победителя на тесте.
chosen = {}
improved_scores = {}
for classification in (True, False):
    base = bases[classification]
    changes = []
    if LAB == 2:
        strengths = [0.01, 0.1, 10.0] if classification else [0.1, 1.0, 10.0, 50.0]
        changes = [('Масштабирование', {'scale': True})]
        changes += [(f'Масштабирование и штраф {v}', {'scale': True, 'strength': v})
                    for v in strengths]
    if LAB == 4:
        alternative = None if classification else 'sqrt'
        changes = [*changes, ('Другой размер подмножества признаков', {'max_features': alternative})]
    candidates = [('Бейзлайн', base)] + [(name, base | patch) for name, patch in changes]
    data, train, test = load_task(classification)
    table, winner, chosen[classification] = validate_candidates(data, train, classification, candidates)
    improved_scores[classification] = evaluate(data, train, test, classification, chosen[classification])
    print('Классификация' if classification else 'Регрессия')
    print(table.round(4).to_string())
    print('Выбор по CV:', winner, chosen[classification])
    print('Тест:', improved_scores[classification])

Классификация
                                mean     std
гипотеза                                    
Бейзлайн                      0.9652  0.0071
Масштабирование               0.9679  0.0073
Масштабирование и штраф 0.01  0.9436  0.0096
Масштабирование и штраф 0.1   0.9631  0.0101
Масштабирование и штраф 10.0  0.9686  0.0058
Выбор по CV: Масштабирование и штраф 10.0 {'scale': True, 'strength': 10.0}
Тест: {'macro_f1': 0.9636319179499313, 'accuracy': 0.9638888888888889}
Регрессия
                                 mean     std
гипотеза                                     
Бейзлайн                      45.0482  1.7725
Масштабирование               45.0482  1.7725
Масштабирование и штраф 0.1   45.0411  1.7765
Масштабирование и штраф 1.0   44.9989  1.8079
Масштабирование и штраф 10.0  45.0604  1.9512
Масштабирование и штраф 50.0  45.4196  2.4104
Выбор по CV: Масштабирование и штраф 1.0 {'scale': True, 'strength': 1.0}
Тест: {'mae': 42.81199941834889, 'rmse': 53.77745406534389, 'r2': 0.4541

## Собственная реализация

Классификатор минимизирует multinomial log-loss с L2-штрафом, градиент вычисляется вручную, L-BFGS-B используется как численный оптимизатор. Регрессор решает задачу наименьших квадратов через NumPy, при ненулевом штрафе добавляет строки для ridge-регрессии.

In [6]:
# Реализуем модель и предсказание средствами NumPy и SciPy.
class OwnLinear:
    def __init__(self, classification, strength):
        self.classification = classification
        self.strength = strength

    def fit(self, X, y):
        X = np.column_stack((np.ones(len(X)), X))
        if not self.classification:
            # Дополнительные строки реализуют L2-штраф без штрафа на свободный член.
            penalty = np.eye(X.shape[1]) * np.sqrt(self.strength)
            penalty[0, 0] = 0
            self.coef = np.linalg.lstsq(
                np.vstack((X, penalty)), np.r_[y, np.zeros(X.shape[1])], rcond=None
            )[0]
            return self

        self.classes, encoded = np.unique(y, return_inverse=True)
        target = np.eye(len(self.classes))[encoded]
        rows = np.arange(len(X))
        alpha = 1 / self.strength

        def objective(flat):
            coefficients = flat.reshape(X.shape[1], len(self.classes))
            logits = X @ coefficients
            log_probabilities = logits - logsumexp(logits, axis=1, keepdims=True)
            loss = -log_probabilities[rows, encoded].sum()
            loss += alpha / 2 * np.sum(coefficients[1:] ** 2)
            gradient = X.T @ (np.exp(log_probabilities) - target)
            gradient[1:] += alpha * coefficients[1:]
            return loss, gradient.ravel()

        result = minimize(
            objective, np.zeros(X.shape[1] * len(self.classes)),
            method="L-BFGS-B", jac=True, options={"maxiter": 5000},
        )
        if not result.success:
            raise RuntimeError(result.message)
        self.coef = result.x.reshape(X.shape[1], len(self.classes))
        return self

    def predict(self, X):
        scores = np.column_stack((np.ones(len(X)), X)) @ self.coef
        return self.classes[scores.argmax(axis=1)] if self.classification else scores

In [7]:
# Сравниваем четыре варианта на одном тесте и сохраняем метрики для итогового ноутбука.
records = []
for classification in (True, False):
    data, train, test = load_task(classification)
    own_base = evaluate(data, train, test, classification, bases[classification], own=True)
    own_improved = evaluate(data, train, test, classification, chosen[classification], own=True)
    scores = {
        "sklearn_base": baseline_scores[classification],
        "sklearn_tuned": improved_scores[classification],
        "own_base": own_base,
        "own_tuned": own_improved,
    }
    task = "classification" if classification else "regression"
    print(task, pd.DataFrame(scores).T.round(4).to_string(), sep="\n")
    split = sha256(np.asarray(test, dtype="<i8").tobytes()).hexdigest()[:16]
    for version, values in scores.items():
        records.append(dict(lab=LAB, task=task, version=version, n_test=len(test), split=split, **values))

result_path = Path("results") / f"lab-{LAB}.json"
result_path.parent.mkdir(exist_ok=True)
result_path.write_text(json.dumps(records, indent=2) + "\n", encoding="utf-8")

classification
               macro_f1  accuracy
sklearn_base     0.9578    0.9583
sklearn_tuned    0.9636    0.9639
own_base         0.9578    0.9583
own_tuned        0.9636    0.9639
regression
                   mae     rmse      r2
sklearn_base   42.7941  53.8534  0.4526
sklearn_tuned  42.8120  53.7775  0.4541
own_base       42.7941  53.8534  0.4526
own_tuned      42.8120  53.7775  0.4541


## Выводы

**Классификация.** MACRO_F1 библиотечной модели улучшилось на тесте (0.9578 → 0.9636). Собственная реализация дала 0.9578 с базовыми параметрами и 0.9636 с настройками, выбранными по CV для sklearn. Разность между собственным и библиотечным вариантами после настройки составила +0.0000.

**Регрессия.** MAE библиотечной модели ухудшилось на тесте (42.7941 → 42.8120). Собственная реализация дала 42.7941 с базовыми параметрами и 42.8120 с настройками, выбранными по CV для sklearn. Разность между собственным и библиотечным вариантами после настройки составила +0.0000.

Результаты относятся к одному отложенному разбиению. Выбранные по CV настройки не гарантируют улучшения на тесте.